# AI-DS-03 — Leak-Free Grouped Release
**Gate M1 · data-v1.0**

This notebook inspects the cleaned metadata available in this checkout, validates records with `dataset.manifest_builder`, partitions eligible rows by genuine `(source_dataset, recording_id)` groups, calculates statistics, and writes release artifacts only after a no-overwrite preflight. A source without genuine group IDs or resolvable audio is excluded and reported; no IDs or durations are inferred.

## 1. Introduction & Objective

The target proportions are 80% Train, 10% Validation, and 10% Test, using deterministic seed **42**. `GroupShuffleSplit` assigns entire composite recording groups `(source_dataset, recording_id)` to a single partition, preventing recordings/sessions from crossing partitions (acoustic leakage). It targets group counts, so actual utterance ratios can differ. Speaker disjointness is not promised. The metadata field `meeting` is never treated as a recording ID.

## 2. Method and data safeguards

Only metadata files found at the candidate locations below are loaded. Source columns are mapped only when present. Missing genuine recording IDs, invalid rows, and missing audio paths remain exclusions; durations are never imputed. Release outputs are created only if their destination paths do not already exist.

In [1]:
from collections import Counter
from pathlib import Path
import hashlib
import json
import math
import sys
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories")
sys.path.insert(0, str(REPO_ROOT / "src"))
from dataset.manifest_builder import validate_and_normalize_records, split_grouped_records

SOURCE_CANDIDATES = {
    "Erzen": [REPO_ROOT / "data/processed/ArzEn/Erzen_cleaned_metadata.csv", REPO_ROOT / "Data/ArzEn_Prepared/Erzen_cleaned_metadata.csv"],
    "Arabic-English": [REPO_ROOT / "data/processed/Arabic-English-ASR/Arabic_English_cleaned_metadata.csv", REPO_ROOT / "Data/Arabic_English/Arabic_English_cleaned_metadata.csv"],
}
print("Repository:", REPO_ROOT)
for source, candidates in SOURCE_CANDIDATES.items():
    print(source, "candidates:", [(str(p.relative_to(REPO_ROOT)), p.is_file()) for p in candidates])


Repository: D:\Gradution Project\airameet-ai-engine
Erzen candidates: [('data\\processed\\ArzEn\\Erzen_cleaned_metadata.csv', False), ('Data\\ArzEn_Prepared\\Erzen_cleaned_metadata.csv', True)]
Arabic-English candidates: [('data\\processed\\Arabic-English-ASR\\Arabic_English_cleaned_metadata.csv', False), ('Data\\Arabic_English\\Arabic_English_cleaned_metadata.csv', True)]


## 3. Input Inspection

This reports the schemas actually loaded, missing-value totals, available source/speaker/group fields, and examples of group values. It does not interpret pre-existing split labels as session identifiers.

In [2]:
source_frames = {}
for source, candidates in SOURCE_CANDIDATES.items():
    found = next((path for path in candidates if path.is_file()), None)
    if found is not None:
        frame = pd.read_csv(found, encoding="utf-8-sig")
        source_frames[source] = (found, frame)
        print(f"{source}: {len(frame)} rows; {len(frame.columns)} columns; file={found.relative_to(REPO_ROOT)}")
        print("columns:", list(frame.columns))
        missing = frame.isna().sum()
        print("nonzero missing values:", missing[missing.gt(0)].to_dict())
        for field in ("recording_id", "meeting", "speaker_id", "speaker", "status"):
            if field in frame:
                vals = frame[field].dropna().astype(str)
                print(f"{field}: {vals.nunique()} unique; sample={vals.drop_duplicates().head(8).tolist()}")
    else:
        print(f"{source}: no metadata file found at declared candidate paths")
if not source_frames:
    raise RuntimeError("No cleaned metadata was found; nothing can be released")


Erzen: 6216 rows; 11 columns; file=Data\ArzEn_Prepared\Erzen_cleaned_metadata.csv
columns: ['utt_id', 'audio_path', 'text', 'split', 'speaker_id', 'recording_id', 'start', 'end', 'duration', 'sample_rate', 'status']
nonzero missing values: {'duration': 7, 'sample_rate': 7}
recording_id: 112 unique; sample=['C03-R01-I01', 'C03-R02-I01', 'C03-R03-I01', 'C03-R04-I01', 'C03-R05-I01', 'C03-R06-I01', 'C03-R07-I01', 'C03-R08-I01']
speaker_id: 40 unique; sample=['I01', 'I02', 'S02', 'S05', 'S07', 'S09', 'S10', 'S12']
status: 2 unique; sample=['success', 'failed']
Arabic-English: 12472 rows; 7 columns; file=Data\Arabic_English\Arabic_English_cleaned_metadata.csv
columns: ['meeting', 'speaker', 'audio_path', 'text', 'start', 'end', 'duration']
nonzero missing values: {}
meeting: 1 unique; sample=['train']
speaker: 1 unique; sample=['unknown']


## 4. Audio path audit

Try the metadata directory and repository root when resolving each existing `audio_path` value. Paths remain untouched on disk. Only paths resolving to files inside this repository can be included in a portable release manifest.

In [3]:
def resolve_audio(raw_path, metadata_path):
    if not isinstance(raw_path, str) or not raw_path.strip():
        return None
    supplied = Path(raw_path.strip())
    candidates = [supplied] if supplied.is_absolute() else [metadata_path.parent / supplied, REPO_ROOT / supplied]
    for candidate in candidates:
        try:
            resolved = candidate.resolve(strict=True)
            resolved.relative_to(REPO_ROOT)
            if resolved.is_file():
                return resolved
        except (OSError, ValueError):
            pass
    return None

audio_audit = {}
for source, (metadata_path, frame) in source_frames.items():
    if "audio_path" not in frame:
        resolved = []
    else:
        resolved = [resolve_audio(value, metadata_path) for value in frame["audio_path"]]
    n_resolved = sum(path is not None for path in resolved)
    audio_audit[source] = resolved
    print(f"{source}: audio paths resolving to in-repository files: {n_resolved}/{len(frame)}")
    examples = frame.loc[[p is None for p in resolved], "audio_path"].head(5).tolist() if "audio_path" in frame else []
    if examples:
        print("unresolved examples:", examples)


Erzen: audio paths resolving to in-repository files: 6209/6216
unresolved examples: ['audio/train/I01-C03-R19_115281-115365.wav', 'audio/train/I01-C03-R25_143651-143753.wav', 'audio/train/I01-C03-R26_122870-123092.wav', 'audio/train/I01-C03-R36_139684-139901.wav', 'audio/train/I02-C03-R16_109379-109505.wav']
Arabic-English: audio paths resolving to in-repository files: 0/12472
unresolved examples: ['audio/train/000001.wav', 'audio/train/000002.wav', 'audio/train/000003.wav', 'audio/train/000004.wav', 'audio/train/000005.wav']


## 5. Validation & Normalization

Map only actual metadata fields to the documented normalized record schema. Arabic-English `meeting` is intentionally not mapped to `recording_id`; its values can be split labels rather than recording/session identifiers. Failed Erzen status rows receive no imputed duration and are passed to validation as missing duration.

In [4]:
candidates = []
candidate_context = []
source_blockers = {}
sources_without_group_ids = set()
for source, (metadata_path, frame) in source_frames.items():
    has_group = "recording_id" in frame.columns
    if not has_group:
        sources_without_group_ids.add(source)
        source_blockers[source] = "No authoritative recording_id field in cleaned metadata; meeting/split labels are not used as group IDs."
    unresolved_count = sum(path is None for path in audio_audit[source])
    if unresolved_count:
        source_blockers[source] = (source_blockers.get(source, "") + f" {unresolved_count}/{len(frame)} audio paths do not resolve to files inside the repository; those rows cannot enter this release.").strip()
    for row_pos, row in enumerate(frame.to_dict(orient="records")):
        raw_audio = row.get("audio_path")
        resolved_audio = audio_audit[source][row_pos] if row_pos < len(audio_audit[source]) else None
        status = row.get("status")
        duration = row.get("duration")
        if isinstance(status, str) and status.strip().casefold() not in {"success", "ok", "valid"}:
            duration = None
        audio_value = str(resolved_audio.relative_to(REPO_ROOT).as_posix()) if resolved_audio else raw_audio
        normalized = {
            "utterance_id": row.get("utterance_id", row.get("utt_id")),
            "audio_path": audio_value,
            "text": row.get("text"),
            "recording_id": row.get("recording_id"),
            "speaker_id": row.get("speaker_id", row.get("speaker")),
            "start": row.get("start"), "end": row.get("end"),
            "duration": duration, "source_dataset": source,
        }
        candidates.append(normalized)
        candidate_context.append({"source": source, "csv_position": len(candidate_context), "audio_ok": resolved_audio is not None})

validation = validate_and_normalize_records(candidates)
rejected_positions = {item.source_row_position for item in validation.rejected_rows}
accepted_records = []
accepted_iter = iter(validation.accepted_records)
for pos, record in enumerate(candidates):
    if pos not in rejected_positions:
        accepted_records.append((pos, next(accepted_iter)))
print("candidate rows:", len(candidates), "accepted:", len(validation.accepted_records), "rejected:", len(validation.rejected_rows))
print("rejection reason counts (reason occurrences):", validation.rejection_counts)
print("bounded rejection examples:", [{"position": r.source_row_position, "source": candidate_context[r.source_row_position]["source"], "utterance_id": r.utterance_id, "reasons": r.reasons} for r in validation.rejected_rows[:8]])

for source in source_frames:
    n = sum(ctx["source"] == source for ctx in candidate_context)
    no_group = "recording_id" in source_frames[source][1].columns
    rejected = sum(ctx["source"] == source and pos in rejected_positions for pos, ctx in enumerate(candidate_context))
    print(f"{source}: rows={n}; validation_rejected={rejected}; genuine recording_id column={no_group}")


candidate rows: 18688 accepted: 6209 rejected: 12479
rejection reason counts (reason occurrences): {'empty_field:duration': 7, 'empty_field:recording_id': 12472, 'empty_field:utterance_id': 12472}
bounded rejection examples: [{'position': 617, 'source': 'Erzen', 'utterance_id': 'I01-C03-R19_115281-115365', 'reasons': ('empty_field:duration',)}, {'position': 777, 'source': 'Erzen', 'utterance_id': 'I01-C03-R25_143651-143753', 'reasons': ('empty_field:duration',)}, {'position': 794, 'source': 'Erzen', 'utterance_id': 'I01-C03-R26_122870-123092', 'reasons': ('empty_field:duration',)}, {'position': 1038, 'source': 'Erzen', 'utterance_id': 'I01-C03-R36_139684-139901', 'reasons': ('empty_field:duration',)}, {'position': 1399, 'source': 'Erzen', 'utterance_id': 'I02-C03-R16_109379-109505', 'reasons': ('empty_field:duration',)}, {'position': 5140, 'source': 'Erzen', 'utterance_id': 'S15-C03-R15_110799-110896', 'reasons': ('empty_field:duration',)}, {'position': 5575, 'source': 'Erzen', 'uttera

## 6. Eligibility & Grouped Splitting

A row can enter a release only if validation accepted it, its audio resolves to a repository file, and its source has a genuine `recording_id` column. If the eligible rows do not provide at least three composite groups, the splitter will stop and no release is written.

In [5]:
eligible_records = []
eligible_context = []
excluded_audio = Counter()
for pos, record in accepted_records:
    ctx = candidate_context[pos]
    if not ctx["audio_ok"]:
        excluded_audio[ctx["source"]] += 1
        continue
    if ctx["source"] in sources_without_group_ids:
        continue
    eligible_records.append(record)
    eligible_context.append(ctx)
print("eligible rows by source:", dict(Counter(ctx["source"] for ctx in eligible_context)))
print("accepted rows excluded for unresolved audio:", dict(excluded_audio))
split_result = None
split_error = None
try:
    split_result = split_grouped_records(eligible_records)
except ValueError as exc:
    split_error = str(exc)
    print("Grouped splitting blocked:", split_error)

if split_result is not None:
    split_records = {"train": split_result.train_records, "val": split_result.validation_records, "test": split_result.test_records}
    for name in ("train", "val", "test"):
        key = "validation" if name == "val" else name
        print(name, {"utterances": split_result.utterance_counts[key], "groups": split_result.group_counts[key], "utterance_ratio": split_result.utterance_ratios[key], "group_ratio": split_result.group_ratios[key]})
    group_sets = {name: {(r["source_dataset"], r["recording_id"]) for r in rows} for name, rows in split_records.items()}
    id_sets = {name: {r["utterance_id"] for r in rows} for name, rows in split_records.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        assert group_sets[a].isdisjoint(group_sets[b]), f"group leakage: {a}/{b}"
        assert id_sets[a].isdisjoint(id_sets[b]), f"utterance leakage: {a}/{b}"
    assert set.union(*id_sets.values()) == {r["utterance_id"] for r in eligible_records}
    print("Leakage assertions passed; speaker disjointness is not asserted.")
    for name, rows in split_records.items():
        print(name, "sources:", dict(Counter(r["source_dataset"] for r in rows)), "speakers:", dict(Counter(r["speaker_id"] or "<missing>" for r in rows)))


eligible rows by source: {'Erzen': 6209}
accepted rows excluded for unresolved audio: {}
train {'utterances': 5454, 'groups': 88, 'utterance_ratio': 0.8784023192140441, 'group_ratio': 0.7857142857142857}
val {'utterances': 502, 'groups': 12, 'utterance_ratio': 0.08085037848284748, 'group_ratio': 0.10714285714285714}
test {'utterances': 253, 'groups': 12, 'utterance_ratio': 0.04074730230310839, 'group_ratio': 0.10714285714285714}
Leakage assertions passed; speaker disjointness is not asserted.
train sources: {'Erzen': 5454} speakers: {'I01': 904, 'I02': 464, 'S02': 114, 'S05': 162, 'S07': 203, 'S09': 127, 'S10': 115, 'S12': 116, 'S14': 99, 'S16': 133, 'S23': 138, 'S27': 157, 'S39': 57, 'S01': 91, 'S03': 110, 'S06': 194, 'S13': 96, 'S20': 85, 'S21': 169, 'S28': 84, 'S29': 106, 'S34': 76, 'S37': 96, 'S38': 90, 'S04': 178, 'S08': 100, 'S15': 116, 'S18': 109, 'S19': 72, 'S22': 112, 'S24': 141, 'S26': 116, 'S30': 99, 'S32': 127, 'S33': 71, 'S35': 94, 'S36': 133}
val sources: {'Erzen': 502} s

## 7. Dataset Statistics

Duration totals use validated `duration` seconds divided by 3,600; distributions include min, quartiles, median, 90th/95th percentiles, max, and mean. Vocabulary is whitespace-tokenized and case-sensitive, with punctuation retained; report total token occurrences and unique tokens.

In [6]:
def duration_stats(rows):
    values = sorted(float(r["duration"]) for r in rows)
    if not values:
        return {"count": 0, "hours": 0.0}
    def quantile(q):
        index = (len(values) - 1) * q
        lo, hi = math.floor(index), math.ceil(index)
        return values[lo] if lo == hi else values[lo] * (hi-index) + values[hi] * (index-lo)
    return {"count": len(values), "total_seconds": sum(values), "hours": sum(values)/3600, "min": values[0], "q25": quantile(.25), "median": quantile(.5), "q75": quantile(.75), "q90": quantile(.9), "q95": quantile(.95), "max": values[-1], "mean": sum(values)/len(values)}

def split_statistics(rows):
    tokens = [token for row in rows for token in row["text"].split()]
    return {"utterances": len(rows), "groups": len({(r["source_dataset"], r["recording_id"]) for r in rows}), "duration": duration_stats(rows), "vocabulary": {"token_count": len(tokens), "unique_token_count": len(set(tokens)), "tokenization": "whitespace; case-sensitive; punctuation retained"}, "sources": dict(sorted(Counter(r["source_dataset"] for r in rows).items())), "speakers": dict(sorted(Counter(r["speaker_id"] or "<missing>" for r in rows).items()))}

statistics = {name: split_statistics(rows) for name, rows in split_records.items()} if split_result else {}
for name, result in statistics.items():
    print(name, json.dumps(result, ensure_ascii=False, indent=2))


train {
  "utterances": 5454,
  "groups": 88,
  "duration": {
    "count": 5454,
    "total_seconds": 35800.819999999956,
    "hours": 9.94467222222221,
    "min": 0.26,
    "q25": 2.84,
    "median": 5.33,
    "q75": 9.1,
    "q90": 13.466999999999999,
    "q95": 16.393499999999996,
    "max": 24.94,
    "mean": 6.564140080674726
  },
  "vocabulary": {
    "token_count": 90843,
    "unique_token_count": 11924,
    "tokenization": "whitespace; case-sensitive; punctuation retained"
  },
  "sources": {
    "Erzen": 5454
  },
  "speakers": {
    "I01": 904,
    "I02": 464,
    "S01": 91,
    "S02": 114,
    "S03": 110,
    "S04": 178,
    "S05": 162,
    "S06": 194,
    "S07": 203,
    "S08": 100,
    "S09": 127,
    "S10": 115,
    "S12": 116,
    "S13": 96,
    "S14": 99,
    "S15": 116,
    "S16": 133,
    "S18": 109,
    "S19": 72,
    "S20": 85,
    "S21": 169,
    "S22": 112,
    "S23": 138,
    "S24": 141,
    "S26": 116,
    "S27": 157,
    "S28": 84,
    "S29": 106,
    "S30": 99

## 8. Release Integrity

Writes deterministic UTF-8 JSONL with records sorted by normalized `utterance_id`; object keys are sorted and JSON separators are fixed. The SHA256 file contains hashes of the exact manifest bytes. `dataset_stats.json` records source scope, exclusions, split ratios, and statistics. Existing targets stop the release before writing.

In [7]:
manifest_dir = REPO_ROOT / "data/manifests"
stats_path = REPO_ROOT / "reports/dataset_stats.json"
target_paths = [manifest_dir / "train.jsonl", manifest_dir / "val.jsonl", manifest_dir / "test.jsonl", manifest_dir / "CHECKSUMS.sha256", stats_path]
release_blockers = list(source_blockers.values())
if split_error:
    release_blockers.append(split_error)
if split_result is None:
    print("No manifests or statistics written because grouped splitting is blocked.")
else:
    manifest_bytes = {}
    for split_name, filename in (("train", "train.jsonl"), ("val", "val.jsonl"), ("test", "test.jsonl")):
        ordered = sorted(split_records[split_name], key=lambda r: r["utterance_id"])
        lines = [json.dumps(r, ensure_ascii=False, sort_keys=True, separators=(",", ":")) for r in ordered]
        manifest_bytes[filename] = (("\n".join(lines) + "\n") if lines else "").encode("utf-8")
    checksums = "".join(f"{hashlib.sha256(manifest_bytes[name]).hexdigest()}  {name}\n" for name in ("train.jsonl", "val.jsonl", "test.jsonl"))
    rejected_by_source = {source: sum(ctx["source"] == source and pos in rejected_positions for pos, ctx in enumerate(candidate_context)) for source in source_frames}
    stats_doc = {"release": "data-v1.0", "seed": 42, "target_ratios": {"train": 0.8, "validation": 0.1, "test": 0.1}, "group_key": ["source_dataset", "recording_id"], "status": "partial" if release_blockers or excluded_audio or any(rejected_by_source.values()) else "complete", "included_sources": sorted({r["source_dataset"] for rows in split_records.values() for r in rows}), "source_blockers": source_blockers, "validation_rejected_rows": rejected_by_source, "accepted_rows_excluded_unresolved_audio": dict(excluded_audio), "splits": {name: {**statistics[name], "utterance_ratio": split_result.utterance_ratios["validation" if name == "val" else name], "group_ratio": split_result.group_ratios["validation" if name == "val" else name]} for name in split_records}, "methodology": {"duration": "validated duration seconds; hours = seconds/3600; linear-interpolated quantiles", "vocabulary": "whitespace tokenization; case-sensitive; punctuation retained", "ratios": "actual utterance and group fractions; row fractions are not guaranteed to equal targets"}}
    stats_bytes = (json.dumps(stats_doc, ensure_ascii=False, sort_keys=True, indent=2) + "\n").encode("utf-8")
    artifact_bytes = {manifest_dir / name: body for name, body in manifest_bytes.items()}
    artifact_bytes[manifest_dir / "CHECKSUMS.sha256"] = checksums.encode("utf-8")
    artifact_bytes[stats_path] = stats_bytes
    existing_paths = [path for path in target_paths if path.exists()]
    if existing_paths:
        if len(existing_paths) != len(target_paths):
            raise FileExistsError("Partial release artifacts exist; refusing to overwrite or complete them")
        mismatches = [str(path.relative_to(REPO_ROOT)) for path, expected in artifact_bytes.items() if path.read_bytes() != expected]
        if mismatches:
            raise FileExistsError("Existing release artifacts differ from deterministic output: " + ", ".join(mismatches))
        print("Existing artifacts inspected and match deterministic output:", [str(path.relative_to(REPO_ROOT)) for path in target_paths])
    else:
        manifest_dir.mkdir(parents=True, exist_ok=True)
        stats_path.parent.mkdir(parents=True, exist_ok=True)
        for path, content in artifact_bytes.items():
            with path.open("xb") as handle:
                handle.write(content)
        print("Created artifacts:", [str(path.relative_to(REPO_ROOT)) for path in target_paths])
    for name in ("train.jsonl", "val.jsonl", "test.jsonl"):
        path = manifest_dir / name
        checksum_line = next(line for line in checksums.splitlines() if line.endswith("  " + name))
        assert hashlib.sha256(path.read_bytes()).hexdigest() == checksum_line.split()[0]
        split_name = "val" if name == "val.jsonl" else name.removesuffix(".jsonl")
        assert sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line) == len(split_records[split_name])
    assert (manifest_dir / "CHECKSUMS.sha256").read_text(encoding="utf-8") == checksums
    assert json.loads(stats_path.read_text(encoding="utf-8"))["release"] == "data-v1.0"
    print("SHA256 checksums:\n" + checksums)
    print("Release status:", stats_doc["status"], "; included sources:", stats_doc["included_sources"])
    if release_blockers:
        print("Blockers:", release_blockers)


Existing artifacts inspected and match deterministic output: ['data\\manifests\\train.jsonl', 'data\\manifests\\val.jsonl', 'data\\manifests\\test.jsonl', 'data\\manifests\\CHECKSUMS.sha256', 'reports\\dataset_stats.json']
SHA256 checksums:
9d9844d8203d6c5425b91a7a326ec0aef132e949f8cf572431dee9ba29522648  train.jsonl
82c0f335fe12314893c7c907a2052525a16a887829bc5b208a78daaf55e80420  val.jsonl
3522252bb27e6ee14a48682306a550458e04fee2d6a6959479189b1228f5bdc2  test.jsonl

Release status: partial ; included sources: ['Erzen']
Blockers: ['7/6216 audio paths do not resolve to files inside the repository; those rows cannot enter this release.', 'No authoritative recording_id field in cleaned metadata; meeting/split labels are not used as group IDs. 12472/12472 audio paths do not resolve to files inside the repository; those rows cannot enter this release.']


## 9. Gate M1 interpretation

A release containing only eligible sources is explicitly a **partial** `data-v1.0` release when any source is blocked or rows are excluded. It must not be described as covering all sources or as a complete Gate M1 release. Group leakage assertions apply to the included records only; no claim of acoustic isolation is possible for sources without authentic recording/session IDs.